# 01 · Data loading + checks
**Owner:** Senanayake · **Branch:** `data/01-loading-checks`

Read all 3 batches from the MATR/Severson `.mat` structs, apply the canonical cell removals + batch1↔batch2 continuation merge, and run the data-quality checks (missing / duplicate / invalid / cycle-order / ranges).


In [ ]:
# --- Colab setup (run first) ---
import os
REPO = '/content/ev-battery-rul-prediction'   # absolute path -> safe to re-run
if not os.path.exists(REPO):
    !git clone https://github.com/Chathumi-Senanayake/ev-battery-rul-prediction.git {REPO}
%cd {REPO}
!git pull -q
!pip install -q -r requirements.txt
import sys
if REPO not in sys.path: sys.path.insert(0, REPO)

# Try Google Drive (data persists across sessions). If auth fails, fall back
# to local Colab storage so you are never blocked.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/ev_battery_data'
except Exception as e:
    print('Drive mount failed -> using local storage:', e)
    DATA_DIR = '/content/ev_battery_data'
os.makedirs(DATA_DIR, exist_ok=True)
print('DATA_DIR =', DATA_DIR)


### Download (run once)
All 3 batch URLs are built into `src/data_loading.py`. Files go to `DATA_DIR`. On local storage they are wiped if the runtime restarts, so re-run this cell after any restart.


In [ ]:
from src.data_loading import DOWNLOAD, MAT_FILENAMES
for b in ['batch1','batch2','batch3']:
    out = os.path.join(DATA_DIR, MAT_FILENAMES[b])
    if os.path.exists(out): print('have', MAT_FILENAMES[b]); continue
    !wget -q -O '{out}' '{DOWNLOAD[b]}' && echo 'downloaded {b}'


### Load + merge
`load_batch(..., lean=True)` (default) keeps summary + curve arrays only, so all 3 batches fit in a standard Colab runtime. If you hit a RAM crash, use **Runtime → Change runtime type → High-RAM**.


In [ ]:
import pickle, gc
from src.data_loading import load_batch, merge_batches, batch_counts
b1 = load_batch(os.path.join(DATA_DIR, MAT_FILENAMES['batch1']), 'b1')
b2 = load_batch(os.path.join(DATA_DIR, MAT_FILENAMES['batch2']), 'b2')
b3 = load_batch(os.path.join(DATA_DIR, MAT_FILENAMES['batch3']), 'b3')
print('raw cells:', len(b1), len(b2), len(b3))
print('expected:', batch_counts(b1, b2, b3))
bat_dict = merge_batches(b1, b2, b3)
print('merged:', len(bat_dict), '(expect 124)')
pickle.dump(bat_dict, open(os.path.join(DATA_DIR,'bat_dict.pkl'),'wb'))
del b1, b2, b3; gc.collect()


### Quality checks


In [ ]:
from src.preprocessing import build_summary_table, quality_report, print_report
df = build_summary_table(bat_dict)
print_report(quality_report(df))
df.head()
